# PyTraverse Übungsbook

Wir arbeiten an ein paar Beispieln wie uns PyTraverse möglicherweise helfen kann

In [ ]:
import pytraverse as t


t.__doc__

## Rekursive Funktionen

## Übungsaufgabe von JupyerNotebook

Hier eine implementierung nur basierend auf Rekursion.
Eure PyTraverse Lösung ist sicherlich übersichtlicher ;)

In [ ]:
def _recursive_test_data_apply(x: list | dict | tuple) -> list | dict | tuple:
    def _int_float_rule(x: int | float) -> float:
        return x * 2

    def _bool_rule(x: bool) -> bool:
        return x

    def _str_rule(x: str) -> str:
        return x + x

    def _list_rule(x: list) -> list:
        return [_recursive_test_data_apply(item) for item in x] * 2

    def _dict_rule(x: dict) -> dict:
        return {_recursive_test_data_apply(k): _recursive_test_data_apply(v) for k, v in x.items()}

    def _tuple_rule(x: tuple) -> tuple:
        return tuple(
            [_recursive_test_data_apply(item) for item in x] + [_recursive_test_data_apply(item) for item in x]
        )

    if isinstance(x, list):
        return _list_rule(x)

    elif isinstance(x, dict):
        return _dict_rule(x)

    elif isinstance(x, tuple):
        return _tuple_rule(x)

    # Da jedes bool auch ein int ist, muss die Abfrage für bool vor der für int/float kommen
    elif isinstance(x, bool):
        return _bool_rule(x)

    elif isinstance(x, (int, float)):
        return _int_float_rule(x)

    elif isinstance(x, str):
        return _str_rule(x)

    else:
        return x

In [ ]:
test_data = [
    ("Wau", "WauWau"),
    (4, 8),
    (-3.5, -7.0),
    (["a", 3], ["aa", 6, "aa", 6]),
    ({"X": 1, 3: ["a"]}, {"XX": 2, 6: ["aa", "aa"]}),
    ((False, False), (False, False, False, False)),
    (True, True),
    (None, None),
]

for data, expected in test_data:
    result = _recursive_test_data_apply(data)
    assert result == expected, f"Der Traverser hat nicht wie erwartet funktioniert: {data=}, {result=}, {expected=}"
print("Alle Tests erfolgreich bestanden!")

## PyTraverse LinkedList

In [ ]:
class LinkedListNode:
    def __init__(self, value, next_node=None):
        self.value = value
        self.next_node = next_node

    def __repr__(self):
        return f"LinkedListNode({self.value}, {self.next_node})"


def concatenate_linked_lists(list1: LinkedListNode | None, list2: LinkedListNode | None) -> LinkedListNode | None:
    # Basisfall: Wenn die erste Liste leer ist, geben wir die zweite Liste zurück
    if list1 is None:
        return list2
    else:
        current = list1
        while current.next_node is not None:
            current = current.next_node
        current.next_node = list2
        return list1


list1 = LinkedListNode(1, LinkedListNode(2, LinkedListNode(3)))
list2 = LinkedListNode(4, LinkedListNode(5))
result = concatenate_linked_lists(list1, list2)
print(result)

Wir werden nun modifikationnen auf diese Linkelist anwenden, und sehen das die rekursiven Funktionen deutlich mehr Aufwand bedeuten, als die Implementierung mit PyTraverse.

In [ ]:
## Sammle alle Werte in einer verketteten Liste in einer Python-Liste
def _recursive_linked_list_to_list(node: LinkedListNode | None) -> list:
    if node is None:
        return []
    else:
        return [node.value] + _recursive_linked_list_to_list(node.next_node)


def _scalar_addition_rule(x: int | float, addend: int | float) -> int | float:
    return x + addend


def add_scalar_to_linked_list(node: LinkedListNode | None, addend: int | float) -> LinkedListNode | None:
    if node is None:
        return None
    else:
        new_value = _scalar_addition_rule(node.value, addend)
        new_next_node = add_scalar_to_linked_list(node.next_node, addend)
        return LinkedListNode(new_value, new_next_node)


linked_list = LinkedListNode(10, LinkedListNode(20, LinkedListNode(30)))
added_list = add_scalar_to_linked_list(linked_list, 2)
print(_recursive_linked_list_to_list(added_list))


Nun was passiert, wenn wir in der LinkedList auch Strings haben ?

In [ ]:
faulty_linked_list = LinkedListNode(1, LinkedListNode("zwei", LinkedListNode(3.0)))
try:
    added_faulty_list = add_scalar_to_linked_list(faulty_linked_list, 2)
except TypeError as e:
    print(f"Fehler beim Hinzufügen: {e}")

Der Fehler ensteht wie folgt: Gegeben das in *_scalar_addition_rule* `x` nun vom Typ `str` ist, inferiert python das mit `+` die String Konkatination gemeint ist.
Jedoch ist `2` ein integer literal, welches nicht mit der `+` operation auf Strings kompatibel ist.

Wir müssen unsere Methode anpassen!

In [ ]:
def _add_scalar_to_linked_list_str_safe(node: LinkedListNode | None, addend: int | float) -> LinkedListNode | None:
    if node is None:
        return None
    else:
        if not isinstance(node.value, (int, float)):
            new_value = node.value
        else:
            new_value = _scalar_addition_rule(node.value, addend)
        new_next_node = _add_scalar_to_linked_list_str_safe(node.next_node, addend)
        return LinkedListNode(new_value, new_next_node)


added_safe_list = _add_scalar_to_linked_list_str_safe(faulty_linked_list, 2)
print(_recursive_linked_list_to_list(added_safe_list))  # Ausgabe: [6, 'zwei', 8.0]

Ihr seht wir mussten die ganze Rekursive funktion erneut schreiben, somit haben wir sehr viel redundanten Code.
Sehen wir uns das mal mit Pytraverse an

In [ ]:
from collections.abc import Callable


@t.singledispatch_traverser
def linked_list_traverser(node: LinkedListNode, traverse: Callable[[object], object]) -> LinkedListNode | None:
    new_value = traverse(node.value)
    new_next_node = traverse(node.next_node)
    return LinkedListNode(new_value, new_next_node)


@t.singledispatch_traverser
def add_two_traverser(x: int | float) -> int | float:
    return x + 2


my_list_traverser = t.sequential(linked_list_traverser, add_two_traverser)

result_traversed = t.traverse(result, my_list_traverser)
faulty_linked_list_traversed = t.traverse(faulty_linked_list, my_list_traverser)
print(result_traversed)
print(faulty_linked_list_traversed)

Nun haben wir also den Code von oben reproduziert, und viel weniger Code schreiben müssen.
Das wird doch langsam nützlich.
Beachtet, dass wir nun eine feste Konstante angeben mussten, und nicht mehr als Parameter nutzen konnten.
Zudem mussten wir nicht für Strings einen Fall schreiben, da falls PyTraverse keine passen registrierte Methode findet, immer das object unverändert zurückgibt.

Als nächstes betrachten wir den Fall, das wir nur in bestimmten Fällen etwas ausführen wollen.
Beispielsweise wollen wir nur für die Elemente an Positionen mit geraden Index etwas ausführen.

In [ ]:
def _add_scalar_to_linked_list_str_safe(
    node: LinkedListNode | None, addend: int | float, index: int = 0
) -> LinkedListNode | None:
    if node is None:
        return None
    else:
        if not isinstance(node.value, (int, float)):
            new_value = node.value
        else:
            # Nur für gerade Indizes die Addition durchführen
            if index % 2 == 0:
                new_value = -_scalar_addition_rule(node.value, addend)
            else:
                new_value = node.value
        new_next_node = _add_scalar_to_linked_list_str_safe(node.next_node, addend, index + 1)
        return LinkedListNode(new_value, new_next_node)


huge_linked_list = LinkedListNode(1, LinkedListNode(2, LinkedListNode(3, LinkedListNode(4, LinkedListNode(5)))))
added_safe_list = _add_scalar_to_linked_list_str_safe(huge_linked_list, 2)
print(_recursive_linked_list_to_list(added_safe_list))

Wie ihr sehen könnt, haben wir nun auf jedes Zweite Element etwas angewendet.
Nun das ganze mit PyTraverse

In [ ]:
ELEMENT_INDEX = t.GlobalVariable[int](
    "ELEMENT_INDEX",
    default=0,
)  # Zählt die aktuellen Indizes während der Traversierung. Spielt also die Rolle des "index" Parameters oben.


@t.singledispatch_traverser
def leaf_count_traverser(node: object, state: t.State) -> tuple[object, t.State]:
    state[ELEMENT_INDEX] += 1
    print(f"Aktueller Index: {state[ELEMENT_INDEX]}")
    return node, state


@leaf_count_traverser.register
def _(node: LinkedListNode) -> LinkedListNode:
    return node


def even_index_skip(state: t.State) -> bool:
    return state[ELEMENT_INDEX] % 2 != 0  # Überspringe, wenn der Index ungerade ist


result_with_state, state = t.traverse_with_state(
    huge_linked_list,
    t.sequential(
        linked_list_traverser,
        t.traverser(add_two_traverser, skip_if=even_index_skip),
        leaf_count_traverser,
    ),
)

print(result_with_state)
print(state[ELEMENT_INDEX])

Abschließend fügen wir die Funktionalität wieder hinzu, mit einem beliebigen Wert die Einträge in der LinkedList zu erhöhen

In [ ]:
ADD_SCALAR = t.GlobalVariable[float](
    "ADD_SCALAR",
    default=2.0,
)  # Zählt die aktuellen Indizes während der Traversierung. Spielt also die Rolle des "index" Parameters oben.

@t.singledispatch_traverser
def add_two_traverser(x: int | float, state: t.State) -> tuple[int | float, t.State]:
    print(f"Füge {state[ADD_SCALAR]} zu {x} hinzu.")
    return x + state[ADD_SCALAR], state


result_with_state, state = t.traverse_with_state(
    huge_linked_list,
    t.sequential(
        linked_list_traverser,
        t.traverser(add_two_traverser, skip_if=even_index_skip),
        leaf_count_traverser,
    )
)

print(result_with_state)
print(state[ELEMENT_INDEX])

## Machine Learning

Wir erstellen jetzt mal ein einfaches Modell, mit kleinem Datensatz.
Wir werden verschiedenste Veränderungen an dem Modell mithilfe von PyTraverse umsetzen.
Schlussendlich werden wir ein Modell erhalten, welches uns aleatorische und epistemische Unsicherheiten anzeigt.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import make_classification as make_classification_dataset
import matplotlib.pyplot as plt

torch.manual_seed(0)
X, y = make_classification_dataset(
    n_samples=100, n_features=2, n_informative=2, n_redundant=0, n_classes=2, random_state=0
)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)


# View the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.title("Klassifikationsdatensatz")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(2, 10)
        self.fc2 = nn.Linear(10, 20)
        self.fc3 = nn.Linear(20, 2)
        self.activation = nn.ReLU()

    def forward(self, x):  # Diese Methode wird aufgerufen, wenn wir das Modell mit Eingabedaten füttern
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        x = self.activation(x)
        x = self.fc3(x)
        return x


#### Training Setup
model = SimpleModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

### Accuracy before training
logits = model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy before training: {accuracy.item()}")

### Training loop
model.train()
for epoch in range(20):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

### Accuracy after training
logits = model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy after training: {accuracy.item()}")

Zum Verständnis:
- Ein `torch.nn.Linear` ist im Prinzip nichts anderes als eine Matrix multiplikation `Wx + b`, wobei `W` die Dimension `output features x input features`  und `b` ein vector der Dimension `output features`
- Die Magie kommt von der nicht linearen Funktion `torch.nn.ReLu`, die für ein gegebenes $f(x) = \begin{cases} x \; \text{if} \; x > 0 \\ 0 \; \text{sonst} \end{cases}$
- In Machine Learning dreht sich viel um die sogenannte Zielfunktion(Lossfunktion), welches zu einem gewissen grad anzeigt wie gut eine Lösung ist. Hier nutzen wir `nn.CrossEntropyLoss()`, die hohe Werte liefert, wenn das Modell nicht die gewünschte Klassen prädiziert.
- Das Lernen von solchen Modellen geschieht durch Gradien Descent, hier mit dem Algorithmus `torch.optim.SGD`, was grundsätzlich $W' = W - \Delta_W$, wobei $\Delta_W$ der Gradient von der (Lossfunktion) bezüglich $W$.

Wieso funktioniert Gradient Descent?

Aus der Mathematik wissen wir das ein Gradient immer in die Richtung `zeigt` sodass der Wert der Funktion erhöht wird.
Intuitiv, gehen wir in die indirekte Richtung (i.e. $-\Delta_W$) so müssen wir zwangsläufig einen geringeren Funktionswerte erhalten.
Natürlich ist dies nur eine Intuition, es gibt verschiedene Probleme die auftreten können, wofür verschiedenste Lösungen vorgeschlagen wurden.
Eine ausführlichere Erklärung findet ihr [hier](https://methpsy.elearning.psych.tu-dresden.de/mediawiki/index.php/Gradient_Descent#:~:text=Unter%20Gradient%20Descent%20versteht%20man,Tal%20gelegenen%20See%20erreichen%20möchte.).

In [ ]:
#### Decision Boundary Visualization
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    grid_logits = model(grid_points)
    grid_probs = torch.softmax(grid_logits, dim=1)
    grid_preds = torch.argmax(grid_probs, dim=1)
Z = grid_preds.reshape(xx.shape)
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.RdBu)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", cmap=plt.cm.RdBu)
plt.title("Entscheidungsgrenze nach dem Training")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### Austauschen der Initialisierungsmethode

Wir betrachten jetzt eine andere Initialisierungsmethode, von neuronalen Netzen.
Die initialierungs methode befüllt die Anfangswerte von `W` und `b`, gemäß eines Verfahrens.
Für interessiert, findet ihr [hier](https://www.geeksforgeeks.org/machine-learning/weight-initialization-techniques-for-deep-neural-networks/) mehr Informationen.
Dafür programmieren wir einen Traverser, welcher bei allen `torch.nn.Linear` die sogenannte [uniform](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.uniform_) Initialisierung durchführt.
Hier werden die Werte zufällig aus dem Interval von [0,1] gewählt.

In [ ]:
@t.singledispatch_traverser
def module_traverser(module: nn.Module, traverse: Callable[[object], object]) -> nn.Module:
    for name, child in module.named_children():
        new_child = traverse(child)
        setattr(module, name, new_child)
    return module


@module_traverser.register
def _linear_weight_init_rule(layer: nn.Linear) -> nn.Linear:
    nn.init.uniform_(layer.weight)
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)
    print(f"Initialisierte Linear Layer mit uniform: {layer}")
    return layer


model = SimpleModel()
initialized_model = t.traverse(model, module_traverser)
print(initialized_model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(initialized_model.parameters(), lr=0.01)


logits = initialized_model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

initialized_model.train()
for epoch in range(20):
    optimizer.zero_grad()
    outputs = initialized_model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

logits = initialized_model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

In [ ]:
# View the decision boundary
import numpy as np

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    grid_logits = initialized_model(grid_points)
    grid_probs = torch.softmax(grid_logits, dim=1)
    grid_preds = torch.argmax(grid_probs, dim=1)
Z = grid_preds.reshape(xx.shape)
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.RdBu)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", cmap=plt.cm.RdBu)
plt.title("Entscheidungsgrenze nach dem Training")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### Austauschen von Activationfunktion

Und als letzten Schritt tauschen wir doch mal die Activation funktion aus, wobei wir es wieder mit Pytraverse machen.
Dabei erweitern wir einfach den Traverser den wir schon haben

In [ ]:
@module_traverser.register
def _activation_function_swap_rule(module: nn.ReLU) -> nn.Module:
    return nn.LeakyReLU()


model = SimpleModel()
swapped_model = t.traverse(model, module_traverser)
print(swapped_model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(swapped_model.parameters(), lr=0.01)


logits = swapped_model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

swapped_model.train()
for epoch in range(20):
    optimizer.zero_grad()
    outputs = swapped_model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

logits = swapped_model(X)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

In [ ]:
# View the decision boundary
import numpy as np

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    grid_logits = swapped_model(grid_points)
    grid_probs = torch.softmax(grid_logits, dim=1)
    grid_preds = torch.argmax(grid_probs, dim=1)
Z = grid_preds.reshape(xx.shape)
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.RdBu)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", cmap=plt.cm.RdBu)
plt.title("Entscheidungsgrenze nach dem Training")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### Uncertainty Aware Model

Nun haben wir kennegelernt, wie wir torch Modelle anpassen können, ohne die original Architectur neu implementieren zu müssen.
Dies ist genau der Zweck von PyTraverse und ein großer Anreiz für Probly.
Nun wollen wir noch abschließend ein Model Uncertainty Aware machen.
Wie ihr aus der ersten Vorlesung euch noch erinnert, basieren diese Ansätze häufig darauf, dass man eine bestehendes Modell anpasst.
Ein simples Beispiel ist der *Dropout* Ansatz.
Hierfür werden wir nach jedem `nn.ReLU` einen `nn.Dropout(p=0.2)` hinzufügen.

In [ ]:
@t.singledispatch_traverser
def _add_dropout_after_activation(module: nn.LeakyReLU | nn.ReLU | nn.Sigmoid) -> nn.Module:
    return nn.Sequential(
        module,
        nn.Dropout(p=0.2),  # Dropout mit einer Wahrscheinlichkeit von 20%
    )


model = SimpleModel()
uncertainty_aware_model = t.traverse(
    model,
    t.sequential(  # Hier verwenden wir t.sequential, um mehrere Traverser nacheinander anzuwenden. Ansonsten würden wir nur den letzten Traverser anwenden.
        module_traverser,
        _add_dropout_after_activation,
    ),
)
print(uncertainty_aware_model)

Intuitiv könnt ihr euch vorstellen, das nach jedem ReLU 20% der Neuronen auf `0` gesetzt werden, somit keine Information mehr weitergeben.
Dadurch wird es ermöglicht dass jeder Aufruf des Models eine unterschiedliche Vorhersage sein kann.
Die Ide dahinter ist, dass wenn das Model unsicher ist, sich eine hohe Varianz in der schlußendlichen Vorhersage ergibt, da das auslassen von Informationen/Neuronen die Entscheidung beeinflussen.
Im Gegensatz dazu, falls wir einen Punkt haben wo das Model sehr sicher ist, so ist es nicht schädlich eine kleine Menge von Information auszulassen, da es auch mit den übrigen 80% immer die selbe Entscheidung trifft.

In [ ]:
N_ITERATIONS = 10  # Durch das Dropout wird das Model stochastisch. Um eine stabilere Vorhersage zu erhalten, mitteln wir über mehrere Vorhersagen.

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(uncertainty_aware_model.parameters(), lr=0.01)


logits = torch.stack([uncertainty_aware_model(X) for _ in range(N_ITERATIONS)]).mean(dim=0)
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

uncertainty_aware_model.train()
for epoch in range(20):
    optimizer.zero_grad()
    outputs = uncertainty_aware_model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

logits = torch.stack([uncertainty_aware_model(X) for _ in range(N_ITERATIONS)]).mean(dim=0)
probs = torch.softmax(logits, dim=-1)
preds = torch.argmax(probs, dim=-1)
accuracy = (preds == y).float().mean()
print(f"Accuracy: {accuracy.item()}")

In [ ]:
# View the decision boundary

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    grid_logits = torch.stack([uncertainty_aware_model(grid_points) for _ in range(N_ITERATIONS)]).mean(dim=0)
    grid_probs = torch.softmax(grid_logits, dim=-1)
    grid_preds = torch.argmax(grid_probs, dim=-1)
Z = grid_preds.reshape(xx.shape)
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.5, cmap=plt.cm.RdBu)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", cmap=plt.cm.RdBu)
plt.title("Entscheidungsgrenze nach dem Training")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

Abschließen können wir uns die verschiedenen Unsicherheiten für dieses Beispiel anschauen.
Hierfür implementieren wir die Entropy basierte Berechnung von aleatorischer und epistemischer Unsicherheiten.
\begin{align*}
    & TU := -\sum_{y \in \mathcal{Y}} \left( \frac{1}{S}\sum_{i \in [S]} p(y | h_i) \right)\cdot \log_2 \left(  \frac{1}{S}\sum_{i \in [S]} p(y | h_i) \right) \\
    & AU := -\frac{1}{S}\sum_{i \in [S]} \left( \sum_{y \in \mathcal{Y}} p(y | h_i)\cdot \log_2 p(y | h_i)) \right) \\
    & EU := TU - AU
\end{align*}
Hierbei steht $S$ für die Anzahl von Aufrufen des Dropout models und $\mathcal{Y}$ sind die Klassen die wir prädizieren.
Somit ist TU die Entropy der gemittelten Prediction, und AU die mittlere Entropy der individuellen Predictionen.
Beachetet eine Prediction ist eine Wahrscheinlichkeitsverteilung über die (beiden) Klassen.
Schlussendlich ist EU die Differenz beider.

In [ ]:
def total_uncertainty(probs: torch.Tensor) -> torch.Tensor:
    """Berechnet die totale Unsicherheit basierend auf den Vorhersage-Wahrscheinlichkeiten.
    Args:
        probs (torch.Tensor): Vorhersage-Wahrscheinlichkeiten des Modells der Form (n_samples, Batchgröße, Anzahl der Klassen).
    """
    mean_probs = probs.mean(dim=0)
    return -torch.sum(mean_probs * torch.log(mean_probs + 1e-10), dim=-1)


def aleatoric_uncertainty(probs: torch.Tensor) -> torch.Tensor:
    """Berechnet die aleatorische Unsicherheit basierend auf den Vorhersage-Wahrscheinlichkeiten.
    Args:
        probs (torch.Tensor): Vorhersage-Wahrscheinlichkeiten des Modells der Form (n_samples, Batchgröße, Anzahl der Klassen).
    """
    entropies = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
    return entropies.mean(dim=0)


def epistemic_uncertainty(probs: torch.Tensor) -> torch.Tensor:
    """Berechnet die epistemische Unsicherheit basierend auf den Vorhersage-Wahrscheinlichkeiten.
    Args:
        probs (torch.Tensor): Vorhersage-Wahrscheinlichkeiten des Modells der Form (n_samples, Batchgröße, Anzahl der Klassen).
    """
    total_unc = total_uncertainty(probs)
    aleatoric_unc = aleatoric_uncertainty(probs)
    return (total_unc - aleatoric_unc).clamp(
        min=0.0
    )  # Aufgrund numerischer Ungenauigkeiten kann es sein, dass total_unc < aleatoric_unc ist


In [ ]:
# View the decision boundary
uncertainty_aware_model.train()

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_points = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    grid_logits = torch.stack([uncertainty_aware_model(grid_points) for _ in range(N_ITERATIONS)])
    grid_probs = torch.softmax(grid_logits, dim=-1)

tu = total_uncertainty(grid_probs).reshape(xx.shape)
au = aleatoric_uncertainty(grid_probs).reshape(xx.shape)
eu = epistemic_uncertainty(grid_probs).reshape(xx.shape)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", zorder=5)
c1 = ax[0].contourf(xx, yy, tu.numpy(), cmap="viridis")
fig.colorbar(c1, ax=ax[0])
ax[0].set_title("Totale Unsicherheit")
ax[1].scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", zorder=5)
c2 = ax[1].contourf(xx, yy, au.numpy(), cmap="viridis")
fig.colorbar(c2, ax=ax[1])
ax[1].set_title("Aleatorische Unsicherheit")
ax[2].scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", zorder=5)
c3 = ax[2].contourf(xx, yy, eu.numpy(), cmap="viridis")
fig.colorbar(c3, ax=ax[2])
ax[2].set_title("Epistemische Unsicherheit")
plt.show()